# 03 · Build the YOLO dataset (stratified split)
**Paper:** Step 3

An image can contain several impurity classes, so a plain random split could leave a class
under- or over-represented in validation. We use a **multi-label stratified split**
(`iterative-stratification`) to divide the 1,000 annotated images into **850 train / 150 val**
while keeping each class's frequency similar in both subsets.

**Inputs** → `data/annotations/` (CVAT YOLO 1.1 export)
**Outputs** → `outputs/yolo_dataset/` (Ultralytics layout + `data.yaml`)

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from PIL import Image, ImageDraw

from requilt.data import (build_label_matrix, label_path_for, read_class_names,
                          read_file_list, read_yolo_labels, yolo_to_xyxy)

ANN = P.annotation_dir
LABELS_DIR = ANN / "obj_train_data"
YOLO_DIR = P.output_dir / "yolo_dataset"

## 1 · Class names and annotated image list

In [ ]:
CLASS_NAMES = read_class_names(ANN / "obj.names")
images = read_file_list(ANN / "train.txt")
print("classes:", CLASS_NAMES)
print(f"annotated images: {len(images)}")                          # expected: 1,000

## 2 · Sanity check: every image has its file and its label

In [ ]:
missing_img = [n for n in images if not (P.quilt_images_dir / n).exists()]
missing_lbl = [n for n in images if not label_path_for(n, LABELS_DIR).exists()]
print(f"missing images: {len(missing_img)}  |  missing labels: {len(missing_lbl)}")   # expected: 0 | 0

## 3 · Multi-label stratified split (85 : 15)
Each image is described by a binary vector that marks which of the four classes it contains.
The split preserves the frequency of every class in both subsets.

In [ ]:
label_matrix = build_label_matrix(images, LABELS_DIR, len(CLASS_NAMES))

splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=cfg.split.val_ratio,
                                            random_state=cfg.seed)
train_idx, val_idx = next(splitter.split(np.array(images), label_matrix))
splits = {"train": [images[i] for i in train_idx], "val": [images[i] for i in val_idx]}

print(f"train: {len(splits['train'])}  |  val: {len(splits['val'])}")   # expected: 850 | 150
print(f"\n{'class':<30}{'all':>6}{'train':>7}{'val':>6}")
for c, name in enumerate(CLASS_NAMES):
    print(f"{name:<30}{label_matrix[:, c].sum():>6}"
          f"{label_matrix[train_idx, c].sum():>7}{label_matrix[val_idx, c].sum():>6}")

# the split shipped in data/splits/ is the one used in the paper - confirm we reproduced it
for name, files in splits.items():
    shipped = P.splits_dir / f"{name}.txt"
    if shipped.exists():
        same = set(read_file_list(shipped)) == set(files)
        print(f"{name} split identical to {shipped.name}: {same}")

## 4 · Write the Ultralytics folder layout and `data.yaml`

In [ ]:
for split, files in splits.items():
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
    for name in files:
        shutil.copy2(P.quilt_images_dir / name, YOLO_DIR / "images" / split / name)
        shutil.copy2(label_path_for(name, LABELS_DIR), label_path_for(name, YOLO_DIR / "labels" / split))
    print(f"[{split}] {len(files)} image/label pairs copied")

names_block = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
(YOLO_DIR / "data.yaml").write_text(
    f"path: {YOLO_DIR.as_posix()}\ntrain: images/train\nval: images/val\n\n"
    f"nc: {len(CLASS_NAMES)}\nnames:\n{names_block}\n"
)
print("data.yaml ->", YOLO_DIR / "data.yaml")